# 05: Read the remembered object at the MiniGrid fork

This notebook runs real `MiniGrid-MemoryS7-v0` episodes with a declared controller. It admits the start-room object once, masks that object from later controller observations, and reads its retained record at the final fork to choose the matching end object.

The corridor geometry and navigation are supplied by the controller. This is a retention experiment with actual environment rewards, not a learned navigation policy or a general MiniGrid benchmark. Missing MiniGrid dependencies stop execution with an installation message; the comparison never substitutes prerecorded rewards.

Saved outputs include each seeded outcome, actual fork-read IDs and payloads, policy/budget comparisons, whole-context controls, operation logs, statistics and source identity.


## 1. Setup


In [ ]:
RUN_MODE = "demo"  # @param ["demo", "paper"]


In [ ]:
import sys
import json
from pathlib import Path

NOTEBOOK_DIR = Path.cwd()
if NOTEBOOK_DIR.name != "notebooks":
    NOTEBOOK_DIR = NOTEBOOK_DIR.parent
sys.path.insert(0, str(NOTEBOOK_DIR))

from _utils import (
    NotebookConfig,
    print_config,
    print_repro_stamp,
    set_all_seeds,
    RunManager,
    bootstrap_ci,
    paired_permutation_test,
    effect_size,
    setup_plotting,
    add_caption,
    COLORS,
    POLICY_COLORS,
)
from _utils.adapters import (
    minigrid_adapter,
    get_env,
    list_envs,
    check_env_available,
    get_install_instructions,
    ReplayTraceEnv,
    get_demo_traces,
    GYMNASIUM_AVAILABLE,
)

from reckon.core.resources import MemoryBudget
from reckon.memory import CausalMemoryManager, PayloadType
from reckon.memory import FIFOPolicy, LRUPolicy, UtilityPolicy, EvidencePerBytePolicy

import matplotlib.pyplot as plt
import numpy as np

setup_plotting()
print("Imports OK!")


In [ ]:
config = NotebookConfig(
    notebook_name="05_minigrid_adapter",
    run_mode=RUN_MODE,
    seed=42,
    seed_count=30 if RUN_MODE == "paper" else 10,
    budget_slots=30,
    task_name="minigrid",
    num_episodes=30 if RUN_MODE == "paper" else 10,
    evidence_weight=0.5,
    custom={"fixture_version": 3, "controller": "masked-full-grid-fixed-corridor"},
)
print_config(config)
print_repro_stamp(config)
set_all_seeds(config.seed)


In [ ]:
MINIGRID_AVAILABLE = minigrid_adapter.check_available()
if not MINIGRID_AVAILABLE:
    raise ImportError(minigrid_adapter.get_install_command())
ENV_NAME = "minigrid_memory"
ENV_ID = "MiniGrid-MemoryS7-v0"
print("Live environment:", ENV_ID)


## 2. The question

[MiniGrid Memory](https://minigrid.farama.org/environments/minigrid/MemoryEnv/) asks the agent to match an object seen in the starting room to one of the two objects beyond a hallway. The cue is an object type, key or ball; both objects are green. It is not a choice between differently coloured doors.

This experiment asks whether a record kept under memory pressure reaches the final decision. The initial cue has assigned utility 2.0; compact observation/action records have 0.1. FIFO, LRU, Utility and Evidence per byte policies compete under the same byte/slot allowances. EvidencePerBytePolicy has no measured effects here and uses its utility branch. Outcome credit is applied after the last decision, so it cannot improve retention earlier in this single episode.


## 3. What the controller can see

The adapter returns a full grid for navigation. On reset, the controller admits only the starting object's type into Reckon as a `CUE`, with `source_step=0`. It then masks the start-object tile in every observation used to choose actions. The controller reads the agent position/direction and the currently visible objects at the fork. It does not read the environment's success position or reach into its grid to recover the answer.

At the fork, a tag query returns the retained initial record. The controller chooses the branch matching that payload. If the record is absent, it chooses north or south using a separate seed-local random generator. After choosing, it completes the turn and forward action without another memory lookup. The transcript records that decision's read IDs and payloads. Keeping the selected action through its execution is distinct from keeping a hidden copy of the original cue.

The 7×7 fixed corridor layout is explicit in the controller; other environment IDs are refused. Wider or random corridors need their own declared controller and checks.


## 4. The execution path

```mermaid
flowchart LR
    A["Reset real environment"] --> B["Write initial object type"]
    B --> C["Mask start tile in controller input"]
    C --> D["Navigate; store compact observations/actions"]
    D --> E["At fork, retrieve initial record"]
    E --> F["Choose matching branch or seeded fallback"]
    F --> G["Execute turn and forward action"]
    G --> H["Observe actual environment reward"]
```

Memory pressure occurs while navigating. At the fork the read happens before the next observation/action writes. A record can therefore be used for the decision and pruned before episode-end credit. Fork retrieval and final record survival are reported separately.


## 5. Experiment design

- Environment: real `MiniGrid-MemoryS7-v0`, with the same generated map and independent fallback seed in each policy arm.
- Samples: ten seeds in demo mode, thirty in paper mode.
- Policies: FIFO, LRU, Utility and Evidence per byte; budgets: 1, 3, 5 and 20 slots, with 500 bytes per slot.
- Outcomes: environment success/reward, whether the cue was retrieved at the fork, final cue survival, stored records and elapsed time.
- Controls: fresh 100-slot memory with and without retrieval, one pair for each seed. The withheld arm still stores records; it is a decision-input control, not a zero-storage arm.

Paper mode joins outcomes by seed, computes a paired permutation test and a normal-approximation interval for each budget, and retains those unadjusted comparisons. It does not infer an effect from the retention/outcome association alone. Byte and slot enforcement follows the [memory manager](../docs/08-Causal-Memory-Manager-Design.md).


## 6. Demo Run


In [ ]:
from _utils.adapters.memory_maze import run_memory_maze
from reckon.metrics.tests.paired import paired_inference

demo_result = run_memory_maze(config.seed, budget_slots=20, policy=FIFOPolicy())
print(
    {
        key: value
        for key, value in demo_result.items()
        if key not in ["events", "transcript"]
    }
)
print(
    "Fork decision:",
    [turn for turn in demo_result["transcript"] if turn["retrieved_ids"]],
)


## 7. Full Run


In [ ]:
# Policy and budget configurations
POLICIES = {
    "FIFO": FIFOPolicy(),
    "LRU": LRUPolicy(),
    "Utility": UtilityPolicy(),
    "Evidence per byte": EvidencePerBytePolicy(config.evidence_weight),
}

BUDGETS = [1, 3, 5, 20]

print(f"Running {config.num_episodes} episodes per configuration")
print(f"Policies: {list(POLICIES.keys())}")
print(f"Budgets: {BUDGETS}")
print(f"Environment: {ENV_NAME}")


In [ ]:
# Run experiments
all_results = []
all_transcripts = []

total_runs = len(POLICIES) * len(BUDGETS) * config.num_episodes
run_count = 0

for policy_name, policy in POLICIES.items():
    for budget in BUDGETS:
        for ep in range(config.num_episodes):
            seed = config.seed + ep
            result = run_memory_maze(
                seed=seed,
                budget_slots=budget,
                policy=policy,
            )

            # Store result (without events for aggregation)
            result_summary = {
                k: v for k, v in result.items() if k not in ["events", "transcript"]
            }
            all_results.append(result_summary)
            all_transcripts.append(
                {
                    "policy": policy_name,
                    "budget": budget,
                    "seed": seed,
                    "transcript": result["transcript"],
                }
            )

            run_count += 1
            if run_count % 50 == 0:
                print(f"Progress: {run_count}/{total_runs}")

print(f"\nCompleted {len(all_results)} runs")


In [ ]:
from statistics import mean, stdev


def scores_for(policy_name, budget, metric):
    return {
        r["seed"]: r[metric]
        for r in all_results
        if r["policy"] == policy_name and r["budget"] == budget
    }


summary = []
for policy in POLICIES.values():
    for budget in BUDGETS:
        row = {"policy": type(policy).__name__, "budget": budget}
        for metric in [
            "success",
            "cue_retrieved_at_fork",
            "initial_cue_retained",
            "steps",
            "records",
        ]:
            row[metric] = mean(
                scores_for(type(policy).__name__, budget, metric).values()
            )
        summary.append(row)
        print(row)


## 8. Results


In [ ]:
# Plot: Success rate and initial cue retention by policy and budget
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Success rate
ax = axes[0]
for policy_name in POLICIES.keys():
    values = [
        list(
            scores_for(type(POLICIES[policy_name]).__name__, budget, "success").values()
        )
        for budget in BUDGETS
    ]
    means = [mean(v) for v in values]
    stds = [stdev(v) for v in values]

    color = POLICY_COLORS.get(policy_name.lower(), COLORS["neutral"])
    ax.errorbar(
        BUDGETS,
        means,
        yerr=stds,
        fmt="o-",
        label=policy_name,
        color=color,
        linewidth=2,
        markersize=8,
        capsize=4,
    )

ax.set_xlabel("Memory Budget (slots)")
ax.set_ylabel("Success Rate")
ax.set_ylim(0, 1.05)
ax.legend()
ax.set_title("Success Rate vs Budget")

# Right: Initial cue retention
ax = axes[1]
for policy_name in POLICIES.keys():
    values = [
        list(
            scores_for(
                type(POLICIES[policy_name]).__name__, budget, "cue_retrieved_at_fork"
            ).values()
        )
        for budget in BUDGETS
    ]
    means = [mean(v) for v in values]
    stds = [stdev(v) for v in values]

    color = POLICY_COLORS.get(policy_name.lower(), COLORS["neutral"])
    ax.errorbar(
        BUDGETS,
        means,
        yerr=stds,
        fmt="o-",
        label=policy_name,
        color=color,
        linewidth=2,
        markersize=8,
        capsize=4,
    )

ax.set_xlabel("Memory Budget (slots)")
ax.set_ylabel("Cue Retrieved at Fork")
ax.set_ylim(0, 1.05)
ax.legend()
ax.set_title("Fork Retrieval vs Budget")

add_caption(
    axes[0],
    f"{ENV_NAME}: actual fork retrieval and environment outcome",
    n=config.num_episodes * len(BUDGETS),
    mode=RUN_MODE,
)

plt.tight_layout()
plt.show()


In [ ]:
paired_comparisons = []
if RUN_MODE == "paper":
    for budget in BUDGETS:
        baseline = scores_for("FIFOPolicy", budget, "success")
        treatment = scores_for("EvidencePerBytePolicy", budget, "success")
        assert baseline.keys() == treatment.keys()
        differences = [
            float(treatment[seed]) - float(baseline[seed]) for seed in sorted(baseline)
        ]
        p_value, interval = paired_inference(
            differences, "permutation", 10000, 0.05, config.seed
        )
        result = {
            "budget": budget,
            "pairs": len(differences),
            "mean_delta": mean(differences),
            "p_value": p_value,
            "normal_interval": list(interval),
            "adjustment": "none",
        }
        paired_comparisons.append(result)
        print(result)


In [ ]:
# Withhold retrieval while keeping ample storage and the same generated map.
context_controls = []
for seed in range(config.seed, config.seed + config.seed_count):
    for use_memory in [True, False]:
        result = run_memory_maze(seed, 100, policy=FIFOPolicy(), use_memory=use_memory)
        assert result["initial_cue_retained"]
        assert not any(event["op_type"] == "prune" for event in result["events"])
        assert result["cue_retrieved_at_fork"] == use_memory
        if use_memory:
            assert result["success"]
        context_controls.append(result)
print(
    "Success with retrieval:",
    mean(r["success"] for r in context_controls if r["use_memory"]),
)
print(
    "Success without retrieval:",
    mean(r["success"] for r in context_controls if not r["use_memory"]),
)
print("The withheld control uses the declared seeded branch fallback.")


In [ ]:
# Analyze tag distribution from a detailed episode
if demo_result.get("events"):
    print("Event Analysis (Demo Episode):")
    print("-" * 40)

    events = demo_result["events"]
    event_types = {}
    for e in events:
        etype = e.get("op_type", "unknown")
        event_types[etype] = event_types.get(etype, 0) + 1

    for etype, count in sorted(event_types.items(), key=lambda x: -x[1]):
        print(f"  {etype}: {count}")
else:
    print("No event log available (demo mode without real environment)")


## 9. Inspect the decision

The saved transcripts connect each fork decision to its retrieved record IDs and object payload. The demo's operation log is separate from those transcripts and covers one episode. Follow the [debugger quick start](../src/reckon/viz/README.md#quick-start) to inspect native write/read/prune IDs.

A final missing record does not prove it was absent at the decision; use `cue_retrieved_at_fork` and the transcript. The imported event log does not contain all comparison episodes or establish a measured graph edge.


## 10. Limits

The controller knows the fixed corridor geometry and uses masked full-grid observations for navigation. It tests whether retained object content reaches one declared decision; it does not evaluate partial-view navigation, a learned RL policy, other MiniGrid tasks or model generality.

The comparison uses assigned priorities. The final outcome updates retained records after the decision but cannot alter earlier retention in that episode. Measured-effect retention and its probe cost remain separate work. Whole-context controls isolate whether the controller receives memory at the fork; they do not estimate an individual record effect under every alternative policy.


## Remove one record at its first read

This separate experiment runs fresh seed-matched episodes under FIFO with enough storage to reach the selected read. It removes one logical record immediately before its first eligible retrieval, verifies matching record state in both arms, and charges both episodes and the full evaluation time. The target is the initial object record, removed just before the controller reads it at the MiniGrid fork. Later replacement records remain allowed. These measurements do not score the main comparison's live store or implement measured retention.

The interval is a normal approximation across paired episode differences; the sign-permutation test is unadjusted across record labels. Four demo pairs and twenty paper-mode pairs demonstrate execution; neither count was selected for statistical power. The probe runs the actual MiniGrid environment with the declared corridor controller. An absent or ambiguous target raises an error rather than becoming a zero effect.

In [ ]:
from _utils.adapters.record_probe import AdapterProbeConfig, evaluate_adapter_record

record_probe = evaluate_adapter_record(
    AdapterProbeConfig("minigrid", budget_slots=100, max_steps=100),
    "first_initial_cue",
    lambda record: "initial_cue" in record.tags,
    n_trials=20 if RUN_MODE == "paper" else 4,
    seed_base=1000,
)
print({key: record_probe[key] for key in ["protocol", "probe_episodes", "probe_elapsed_s", "confidence_interval"]})
print({key: value for key, value in record_probe["evaluation"].items() if key != "trials"})

## Export Artifacts


In [ ]:
run = RunManager(config)
run.save_config()
if record_probe is not None:
    (run.run_dir / "record_probe.json").write_text(json.dumps(record_probe, indent=2) + "\n")
run.save_metrics(all_results)
run.save_jsonl(all_transcripts, "transcripts.jsonl")
run.save_jsonl(context_controls, "context_controls.jsonl")
run.save_jsonl(paired_comparisons, "paired_comparisons.jsonl")

# Save event logs from demo episode
if demo_result.get("events"):
    run.save_logs(demo_result["events"])

# Save figures
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Success rate
ax = axes[0]
for policy_name in POLICIES.keys():
    means = [
        mean(
            scores_for(type(POLICIES[policy_name]).__name__, budget, "success").values()
        )
        for budget in BUDGETS
    ]
    color = POLICY_COLORS.get(policy_name.lower(), COLORS["neutral"])
    ax.plot(BUDGETS, means, "o-", label=policy_name, color=color)
ax.set_xlabel("Budget")
ax.set_ylabel("Success Rate")
ax.legend()

# Cue retention
ax = axes[1]
for policy_name in POLICIES.keys():
    means = [
        mean(
            scores_for(
                type(POLICIES[policy_name]).__name__, budget, "cue_retrieved_at_fork"
            ).values()
        )
        for budget in BUDGETS
    ]
    color = POLICY_COLORS.get(policy_name.lower(), COLORS["neutral"])
    ax.plot(BUDGETS, means, "o-", label=policy_name, color=color)
ax.set_xlabel("Budget")
ax.set_ylabel("Cue Retrieved at Fork")
ax.legend()

run.save_figure(fig, "minigrid_metrics")
plt.close(fig)

# Save environment metadata
env_meta = {
    "env_name": ENV_NAME,
    "env_id": ENV_ID,
    "minigrid_available": MINIGRID_AVAILABLE,
    "gymnasium_available": GYMNASIUM_AVAILABLE,
    "policies": list(POLICIES.keys()),
    "budgets": BUDGETS,
}
with open(run.run_dir / "env_meta.json", "w") as f:
    json.dump(env_meta, f, indent=2)

# Source and backend identity accompany the actual run.
import hashlib
import importlib.metadata
import subprocess

source_paths = [
    NOTEBOOK_DIR / "_utils/adapters/record_probe.py",
    NOTEBOOK_DIR / "_utils/adapters/episode_memory.py",
    NOTEBOOK_DIR / "05_rl_env_adapter_minigrid.ipynb",
    NOTEBOOK_DIR / "_utils/adapters/memory_maze.py",
    NOTEBOOK_DIR / "_utils/adapters/minigrid_adapter.py",
    NOTEBOOK_DIR / "_utils/artifacts.py",
]
source = {
    "run_mode": RUN_MODE,
    "backend_versions": {
        name: importlib.metadata.version(name)
        for name in ["minigrid", "gymnasium", "pygame-ce"]
    },
    "uncommitted_source": bool(
        subprocess.run(
            [
                "git",
                "status",
                "--porcelain",
                "--",
                *[str(path) for path in source_paths],
            ],
            capture_output=True,
            text=True,
            check=True,
        ).stdout
    ),
    "sha256": {
        str(path.relative_to(NOTEBOOK_DIR)): hashlib.sha256(
            path.read_bytes()
        ).hexdigest()
        for path in source_paths
    },
}
(run.run_dir / "source.json").write_text(json.dumps(source, indent=2) + "\n")
run.write_summary(
    {
        "run_mode": RUN_MODE,
        "environment": ENV_ID,
        "comparison_episodes": len(all_results),
        "context_control_episodes": len(context_controls),
        "record_probe_episodes": record_probe["probe_episodes"] if record_probe else 0,
        "record_probe_elapsed_s": record_probe["probe_elapsed_s"] if record_probe else 0,
        "record_probe_delta": record_probe["evaluation"]["delta"] if record_probe else None,
        "record_probe_available": record_probe is not None,
        "inspection_episodes": 1,
        "retrieval_control_success": mean(
            r["success"] for r in context_controls if r["use_memory"]
        ),
        "withheld_control_success": mean(
            r["success"] for r in context_controls if not r["use_memory"]
        ),
    }
)
run.print_artifacts()
